In [ ]:
!pip -q install -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 41.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
from peft import PeftModel

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MODEL_DIR = "/content/qwen25_3b_ielts_4criteria/best_model"   # sửa lại đường dẫn của bạn
MAX_LENGTH = 1024

TARGET_COLS = ["TR", "CC", "LR", "GRA"]

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# config
config = AutoConfig.from_pretrained(BASE_MODEL)
config.num_labels = 4
config.problem_type = "regression"
config.pad_token_id = tokenizer.pad_token_id

# 4-bit nếu muốn nhẹ VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    ignore_mismatched_sizes=True,
)

# gắn adapter
model = PeftModel.from_pretrained(
    base_model,
    MODEL_DIR,
    low_cpu_mem_usage=True,
)

model.eval()
print("Loaded successfully.")
print("num_labels:", model.config.num_labels)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-3B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded successfully.
num_labels: 4


In [ ]:
def round_to_half(x):
    return np.round(x * 2) / 2

def build_input_text(prompt: str, essay: str) -> str:
    prompt = str(prompt).strip()
    essay = str(essay).strip()
    return f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"


@torch.no_grad()
def predict_scores(prompt: str, essay: str):
    text = build_input_text(prompt, essay)

    inputs = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt"
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model(**inputs)
    raw_logits = outputs.logits.detach().float().cpu().numpy()[0]   # shape (4,)

    # PHẢI khớp với compute_metrics lúc train
    preds_band = raw_logits * 9.0
    preds_band = np.clip(preds_band, 0.0, 9.0)
    preds_half = round_to_half(preds_band)

    result = {col: float(score) for col, score in zip(TARGET_COLS, preds_half)}
    result["overall_raw"] = float(np.mean(preds_band))
    result["overall"] = float(round_to_half(np.mean(preds_band)))

    return result

In [ ]:
sample_prompt = "Details of politicians’ private lives should not be published in newspapers. To what extent do you agree or disagree?"
sample_essay = """
It is thought that the information regarding politicians’ personal lives should not be shared in print media. This essay strongly agrees with this suggestion because publishing these details could be harmful to their families, and obtaining this type of information might require breaking the law.

First and foremost, what makes that the details related to private aspects of politicians’ lives should not be shared in newspapers is that it could be harmful not only to these individuals but also to their families. This is because revealing some details from their personal lives could expose them to unwanted comments or allegations, which might lead to a great deal of distress. In Poland, for instance, in 2015, the vice-prime minister committed suicide due to not handling the pressure caused by the paparazzi invading his and his family’s private life.

Furthermore, obtaining this type of information, in most cases, means breaking the law. This is because the right to privacy is one of the most fundamental policies in society, and anyone who wants to access the lives of politicians must obtain their consent. However, not only are paparazzi hired to invade properties belonging to politicians to take photos without their permission, but also politicians’ colleagues and relatives are bribed to share confidential facts from their lives. For instance, an accident in which Princess Diana was killed was partly caused by the paparazzi who followed her car, trying to take photos of her and her boyfriend against their will.

In conclusion, I strongly support the suggestion that politicians’ lives should not be subject to the interest of newspapers because revealing personal facts from politicians lives could destroy their family life and the process of obtaining these details often required wrongdoing.
"""

pred = predict_scores(sample_prompt, sample_essay)
print(pred)

{'TR': 7.0, 'CC': 6.5, 'LR': 6.0, 'GRA': 5.0, 'overall_raw': 6.076887607574463, 'overall': 6.0}


In [ ]:
import numpy as np
import pandas as pd
import torch

TARGET_COLS = ["TR", "CC", "LR", "GRA"]

def round_to_half(x):
    return np.round(x * 2) / 2

def build_input_text(prompt: str, essay: str) -> str:
    prompt = str(prompt).strip()
    essay = str(essay).strip()
    return f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"

@torch.no_grad()
def predict_debug(prompt: str, essay: str):
    text = build_input_text(prompt, essay)

    inputs = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt"
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model(**inputs)
    raw_logits = outputs.logits.detach().float().cpu().numpy()[0]   # shape (4,)

    scaled = raw_logits * 9.0
    clipped = np.clip(scaled, 0.0, 9.0)
    rounded = round_to_half(clipped)

    debug_df = pd.DataFrame({
        "criterion": TARGET_COLS,
        "raw_logit": raw_logits,
        "scaled_x9": scaled,
        "clipped_0_9": clipped,
        "rounded_0_5": rounded,
    })

    return debug_df

In [ ]:
debug_df = predict_debug(sample_prompt, sample_essay)
print(debug_df)

  criterion  raw_logit  scaled_x9  clipped_0_9  rounded_0_5
0        TR   1.109375   9.984375     9.000000          9.0
1        CC   0.769531   6.925781     6.925781          7.0
2        LR   0.785156   7.066406     7.066406          7.0
3       GRA   0.283203   2.548828     2.548828          2.5
